# AI Infrastructure Stock Analysis Pipeline

## Abstract

This project is an end-to-end **data analytics and machine learning pipeline** focused on the stock market. It extracts historical financial data, engineers domain-specific features, performs statistical analysis, and applies machine learning to forecast future price movements.

The workflow utilizes:

- **Python** for data pipeline automation and machine learning  
- **R** for advanced statistical time-series analysis  
- **Power BI** for building an interactive executive dashboard  

The primary objective is to:

- Evaluate financial risk  
- Analyze historical return metrics  
- Predict whether a stock trend is **bullish** (upward) or **bearish** (downward)  

---

## Pipeline Overview

| Stage | Description |
|------|------------|
| **1. Data Acquisition** | Fetch 10-year OHLCV data for AI-sector stocks |
| **2. Data Cleaning** | Standardize structure and remove unnecessary fields |
| **3. Data Validation** | Ensure data quality and logical consistency |
| **4. Feature Engineering** | Generate trend, volatility, and return indicators |
| **5. Target Variable** | Label data for bullish/bearish prediction |


# Stage 1 — Data Acquisition & Initial Formatting

This stage focuses on collecting historical stock data for selected companies representing the AI infrastructure ecosystem.

### Selected Stock Categories

- **Chip Designers:** NVDA, AMD  
- **Semiconductor Foundry:** TSM  
- **Hyperscalers:** MSFT, GOOGL  

### Key Objectives

- Retrieve **10 years of daily OHLCV data**
- Combine all tickers into a **single unified dataset**
- Ensure proper **chronological sorting**
- Store raw data as a **checkpoint for reproducibility**


In [1]:
import yfinance as yf
import pandas as pd
import numpy as np

In [2]:
# Define the AI Infrastructure stock basket.
# These five tickers represent the key layers of the AI supply chain:
# chip designers (NVDA, AMD), foundry (TSM), and hyperscalers (MSFT, GOOGL).
tickers = ["NVDA", "AMD", "TSM", "MSFT", "GOOGL"]

# Container to collect each ticker's DataFrame before combining them.
all_data = []

In [3]:
# Download 10 years of daily OHLCV history for each ticker.
# yfinance automatically adjusts prices for stock splits and dividends,
# ensuring a clean, comparable price series across the full history.
for ticker in tickers:
    print(f"Downloading 10-year historical data for {ticker}...")
    stock = yf.Ticker(ticker)

    df_ticker = stock.history(period="10y")

    # Promote the DatetimeIndex into a regular column for easier manipulation.
    df_ticker = df_ticker.reset_index()

    # Tag each row with its source ticker so rows remain identifiable after merging.
    df_ticker['Ticker'] = ticker

    all_data.append(df_ticker)

print("\nAll tickers downloaded successfully.")


All tickers downloaded successfully.


In [4]:
# Concatenate all individual ticker DataFrames into one unified DataFrame.
# Without this step, downstream code would only operate on the last ticker
# from the loop (GOOGL), silently discarding the other four stocks.
df = pd.concat(all_data, ignore_index=True)

# Sort chronologically within each ticker to ensure rolling calculations
# and shift operations produce correct results.
df = df.sort_values(['Ticker', 'Date']).reset_index(drop=True)

print(f"Combined dataset shape : {df.shape}")
print(f"Tickers present        : {df['Ticker'].unique().tolist()}")
print(f"Date range             : {df['Date'].min().date()} → {df['Date'].max().date()}")

Combined dataset shape : (12570, 9)
Tickers present        : ['AMD', 'GOOGL', 'MSFT', 'NVDA', 'TSM']
Date range             : 2016-05-02 → 2026-04-30


In [5]:
# Persist the raw, unprocessed data as a checkpoint before any transformations.
# This preserves the original download and makes it easy to restart from here
# without hitting the yfinance API again.
raw_file = "raw_df.csv"
df.to_csv(raw_file, index=False)
print(f"Raw data saved → {raw_file}")

Raw data saved → raw_df.csv


# Stage 2 — Data Cleaning & Preprocessing

### Objectives

- Remove timezone information from the `date` column  
- Drop unnecessary columns such as:
  - `Dividends`
  - `Stock Splits`  
- Standardize all column names using **snake_case convention**

### Outcome

A clean and consistent dataset ready for validation and analysis across:

- Python  
- R  
- Power BI  


In [6]:
# Power BI and R's lubridate both require timezone-naive datetime objects.
# tz_localize(None) removes the UTC offset without shifting the timestamp value.
df['Date'] = pd.to_datetime(df['Date']).dt.tz_localize(None)

In [7]:
# Retain only the core OHLCV columns plus the Ticker identifier.
# 'Dividends' and 'Stock Splits' are dropped because prices are already
# adjusted for these events by yfinance.
df = df[['Date', 'Ticker', 'Open', 'High', 'Low', 'Close', 'Volume']]

In [8]:
# Rename all columns to snake_case for consistent use across Python, R, and Power BI.
rename_map = {
    'Date':   'date',
    'Ticker': 'ticker',
    'Open':   'open_price',
    'High':   'high_price',
    'Low':    'low_price',
    'Close':  'close_price',
    'Volume': 'trading_volume'
}

df = df.rename(columns=rename_map)

print("Columns after renaming:")
print(df.columns.tolist())

Columns after renaming:
['date', 'ticker', 'open_price', 'high_price', 'low_price', 'close_price', 'trading_volume']


# Stage 3 — Data Quality & Validation

Before feature engineering, three essential validation checks are performed:

### 1. Null Value Check
Ensure no missing values exist in OHLCV data.

### 2. OHLC Logical Consistency

Each row must satisfy:

- `high ≥ max(open, close)`  
- `low ≤ min(open, close)`  
- `high ≥ low`  

Any violation indicates invalid financial data.

### 3. Duplicate Check

- Each `(date, ticker)` pair must be unique  
- Ensures one record per stock per trading day  


In [9]:
# Check 1: Missing values.
# All raw OHLCV columns should be fully populated at this stage.
print("── Null Check ──")
null_counts = df.isnull().sum()
print(null_counts)

if null_counts.sum() == 0:
    print("\n✓ No missing values found.")
else:
    print("\n⚠ Missing values detected — review before proceeding.")


── Null Check ──
date              0
ticker            0
open_price        0
high_price        0
low_price         0
close_price       0
trading_volume    0
dtype: int64

✓ No missing values found.


In [10]:
# Check 2: OHLC logic invariants.
# A valid candlestick must satisfy ALL of the following:
#   high  ≥  max(open, close)   — high is the session ceiling
#   low   ≤  min(open, close)   — low is the session floor
#   high  ≥  low                — trivially required
#
# NOTE: The original notebook was missing the `low > close_price` condition.
# A close price below the day's low is equally invalid and must be caught.
print("── OHLC Logic Check ──")
price_errors = df[
    (df['high_price'] < df['low_price'])   |   # high below low
    (df['high_price'] < df['close_price']) |   # close above high
    (df['high_price'] < df['open_price'])  |   # open above high
    (df['low_price']  > df['close_price']) |   # close below low  ← added
    (df['low_price']  > df['open_price'])      # open below low
]

if not price_errors.empty:
    print(f"⚠ {len(price_errors)} rows with invalid OHLC relationships:")
    print(price_errors[['date', 'ticker', 'open_price',
                         'high_price', 'low_price', 'close_price']].head())
else:
    print("✓ OHLC logic check passed — all rows are valid.")

── OHLC Logic Check ──
⚠ 1 rows with invalid OHLC relationships:
            date ticker  open_price  high_price  low_price  close_price
10397 2017-09-07    TSM   30.642863   30.889717  30.593491    30.889717


In [11]:
# Check 3: Duplicate rows.
# Each (date, ticker) pair must be unique — a stock has exactly one price
# record per trading day.
print("── Duplicate Check ──")
duplicates = df.duplicated(subset=['date', 'ticker']).sum()

if duplicates > 0:
    print(f"⚠ {duplicates} duplicate (date, ticker) rows found.")
else:
    print("✓ No duplicates found.")

── Duplicate Check ──
✓ No duplicates found.


# Stage 4 — Feature Engineering

Features are grouped into three analytical categories:

| Category | Features | Purpose |
|----------|---------|--------|
| **Trend / Smoothing** | EMA-20, SMA-50, SMA-200 | Identify direction and momentum |
| **Risk / Volatility** | Std Dev-20, Bollinger Bands | Measure uncertainty and risk |
| **Returns** | Daily, Log, Cumulative, Monthly | Analyze profitability |

> All calculations are performed **per ticker** using grouped operations to prevent data leakage.


## 4.1 — Trend Indicators (Smoothing Metrics)

- **EMA-20:** Fast-moving average emphasizing recent prices  
- **SMA-50:** Medium-term trend used by traders  
- **SMA-200:** Long-term institutional benchmark  

### Notes

- Initial rows contain `NaN` due to insufficient historical data  
- This behavior is expected and correct  


In [12]:
# EMA-20: Fast exponential moving average.
# adjust=False uses the recursive formula:
#   EMA_t = alpha * Price_t + (1 - alpha) * EMA_(t-1)
df['ema_20'] = df.groupby('ticker')['close_price'].transform(
    lambda x: x.ewm(span=20, adjust=False).mean()
)

# SMA-50: Medium-term simple moving average.
df['sma_50'] = df.groupby('ticker')['close_price'].transform(
    lambda x: x.rolling(window=50).mean()
)

# SMA-200: Long-term simple moving average (the institutional benchmark).
df['sma_200'] = df.groupby('ticker')['close_price'].transform(
    lambda x: x.rolling(window=200).mean()
)

print("✓ Trend indicators calculated.")

✓ Trend indicators calculated.


## 4.2 — Volatility & Risk Indicators

- **20-Day Standard Deviation:** Measures price variability  
- **Bollinger Bands:**  
  - Middle Band → 20-day SMA  
  - Upper/Lower Bands → ±2 standard deviations  

### Interpretation

- Wider bands → Higher volatility  
- Narrow bands → Lower volatility  


In [13]:
# 20-day rolling standard deviation of close price.
df['std_dev_20'] = df.groupby('ticker')['close_price'].transform(
    lambda x: x.rolling(window=20).std()
)

# Bollinger Band middle line — identical to SMA-20, kept explicit for Power BI clarity.
df['bollinger_mid'] = df.groupby('ticker')['close_price'].transform(
    lambda x: x.rolling(window=20).mean()
)

# Upper and lower bands sit 2 standard deviations away from the middle band.
df['bollinger_upper'] = df['bollinger_mid'] + (df['std_dev_20'] * 2)
df['bollinger_lower'] = df['bollinger_mid'] - (df['std_dev_20'] * 2)

print("✓ Volatility & risk indicators calculated.")

✓ Volatility & risk indicators calculated.


In [19]:
# -----------------------------
# 2. Rolling Volatilit<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# -----------------------------


# 20-day rolling volatility (short-term)
df['rolling_vol_20d'] = (
    df.groupby('ticker')['daily_return']
    .transform(lambda x: x.rolling(window=20).std() * np.sqrt(252))
)

# 60-day rolling volatility (medium-term)
df['rolling_vol_60d'] = (
    df.groupby('ticker')['daily_return']
    .transform(lambda x: x.rolling(window=60).std() * np.sqrt(252))
)

#1-year rolling 
df['rolling_vol_252d'] = (
    df.groupby('ticker')['daily_return']
    .transform(lambda x: x.rolling(window=252).std() * np.sqrt(252))
)

In [21]:
# Max Drawdown 
df['cum_max'] = df.groupby('ticker')['close_price'].cummax()
df['drawdown'] = df['close_price'] / df['cum_max'] - 1

In [22]:
# -----------------------------
# 2. Compute Running Peak
# -----------------------------
df['running_peak'] = (
    df.groupby('ticker')['cum_return']
    .cummax()
)


## 4.3 — Return Metrics

- **Daily Return:** Percentage change between consecutive days  
- **Log Return:**  
  - Formula: `ln(P_t / P_{t-1})`  
  - Preferred for machine learning and statistical models  
- **Cumulative Return:** Total compounded growth over time  
- **Monthly Return:** Approximation using 21 trading days  


In [23]:
# Daily simple return: (Close_today - Close_yesterday) / Close_yesterday
df['daily_return'] = df.groupby('ticker')['close_price'].pct_change()

# Log return: ln(Close_today / Close_yesterday)
# Preferred for ML features because it is symmetric and additively composable.
df['log_return'] = df.groupby('ticker')['close_price'].transform(
    lambda x: np.log(x / x.shift(1))
)

# Cumulative return since day 1 for each ticker.
# Uses the compound growth formula: product of (1 + daily_return) minus 1.
df['cum_return'] = df.groupby('ticker')['daily_return'].transform(
    lambda x: (1 + x).cumprod() - 1
)

# Approximate monthly return using a 21 trading-day rolling window.
df['monthly_return'] = df.groupby('ticker')['close_price'].pct_change(periods=21)

# The first row(s) of each ticker will be NaN because there is no prior close.
# Fill with 0 (no return on the first observable day).
cols_to_fix = ['daily_return', 'log_return', 'cum_return', 'monthly_return']
df[cols_to_fix] = df[cols_to_fix].fillna(0)

print("✓ Return metrics calculated.")

✓ Return metrics calculated.


# Stage 5 — Target Variable Construction

The problem is framed as a **binary classification task**:

> **Will tomorrow’s closing price be significantly higher than today’s?**

### Target Definition

- **Bullish (1):**
  - Tomorrow’s close > Today’s close by more than **0.5%**

- **Bearish / Neutral (0):**
  - Price remains flat or decreases  

### Important Considerations

- The last row of each ticker is removed due to missing future data  
- Class distribution is evaluated to detect imbalance  

### Why 0.5% Threshold?

- Filters out insignificant price movements  
- Focuses the model on meaningful trends  


In [24]:
# Look one day into the future by shifting the close price up by one row (per ticker).
# The last trading day of each ticker will produce NaN here — handled below.
df['tomorrow_close'] = df.groupby('ticker')['close_price'].shift(-1)

# Label as Bullish (1) only if tomorrow's close exceeds today's by more than 0.5%.
df['target'] = (df['tomorrow_close'] > df['close_price'] * 1.005).astype(int)

In [25]:
# Drop the last row of each ticker where tomorrow_close is NaN.
# With 5 tickers, this removes exactly 5 rows — one per ticker.
# These rows have no valid target and must be excluded before any ML training
# to avoid leakage or silent label errors.
rows_before = len(df)
df = df.dropna(subset=['tomorrow_close']).reset_index(drop=True)
print(f"Dropped {rows_before - len(df)} terminal rows (one per ticker) with no next-day target.")
print(f"Final dataset shape: {df.shape}")

Dropped 5 terminal rows (one per ticker) with no next-day target.
Final dataset shape: (12565, 26)


In [26]:
# Class distribution check.
# Understanding class balance is critical before training any classifier.
# A heavily skewed target biases the model toward the majority class.
print("── Target Distribution ──")
dist = df['target'].value_counts(normalize=True)
dist.index = dist.index.map({0: 'Bearish (0)', 1: 'Bullish (1)'})
print(dist.to_string())

imbalance_ratio = dist.max() / dist.min()
print(f"\nImbalance ratio: {imbalance_ratio:.2f}x")

if imbalance_ratio > 1.5:
    print(
        "⚠ Moderate class imbalance detected.\n"
        "  Consider: SMOTE oversampling, class_weight='balanced',\n"
        "  or use precision–recall AUC instead of accuracy when evaluating classifiers."
    )
else:
    print("✓ Classes are reasonably balanced.")


── Target Distribution ──
target
Bearish (0)    0.58846
Bullish (1)    0.41154

Imbalance ratio: 1.43x
✓ Classes are reasonably balanced.


In [27]:
# Sanity check: inspect the tail of one ticker to confirm that the target label
# correctly reflects the relationship between close_price and tomorrow_close.
nvda_tail = df[df['ticker'] == 'NVDA'].tail(3)

if nvda_tail.empty:
    print("⚠ No NVDA rows found — verify the ticker loop and concat ran correctly.")
else:
    print("NVDA — Last 3 rows (target alignment check):")
    print(nvda_tail[['date', 'ticker', 'close_price', 'tomorrow_close', 'target']].to_string(index=False))

NVDA — Last 3 rows (target alignment check):
      date ticker  close_price  tomorrow_close  target
2026-04-27   NVDA   216.610001      213.169998       0
2026-04-28   NVDA   213.169998      209.250000       0
2026-04-29   NVDA   209.250000      199.970001       0


In [28]:
price_cols  = ['open_price', 'high_price', 'low_price', 'close_price',
               'tomorrow_close', 'ema_20', 'sma_50', 'sma_200',
               'std_dev_20', 'bollinger_mid', 'bollinger_upper', 'bollinger_lower']

return_cols = ['daily_return', 'log_return', 'cum_return', 'monthly_return']

df[price_cols]  = df[price_cols].round(2)
df[return_cols] = df[return_cols].round(4)

print("✓ Final rounding applied.")

✓ Final rounding applied.


In [29]:
print("── Null Check ──")
null_counts = df.isnull().sum()
print(null_counts)

if null_counts.sum() == 0:
    print("\n✓ No missing values found.")
else:
    print("\n⚠ Missing values detected — review before proceeding.")

── Null Check ──
date                   0
ticker                 0
open_price             0
high_price             0
low_price              0
close_price            0
trading_volume         0
ema_20                 0
sma_50               245
sma_200              995
std_dev_20            95
bollinger_mid         95
bollinger_upper       95
bollinger_lower       95
daily_return           0
log_return             0
cum_return             0
monthly_return         0
rolling_vol_20d       95
rolling_vol_60d      295
rolling_vol_252d    1255
cum_max                0
drawdown               0
running_peak           0
tomorrow_close         0
target                 0
dtype: int64

⚠ Missing values detected — review before proceeding.


In [31]:
# Drop warm-up rows where sma_200 is NaN.
# sma_200 requires the largest window (200 days), so dropping on it
# automatically clears all smaller-window NaNs (sma_50, std_dev_20,
# bollinger bands) at the same time — no extra steps needed.

rows_before = len(df)
df = df.dropna()

print("── Null Check after drop ──")
print(df.isnull().sum())

── Null Check after drop ──
date                0
ticker              0
open_price          0
high_price          0
low_price           0
close_price         0
trading_volume      0
ema_20              0
sma_50              0
sma_200             0
std_dev_20          0
bollinger_mid       0
bollinger_upper     0
bollinger_lower     0
daily_return        0
log_return          0
cum_return          0
monthly_return      0
rolling_vol_20d     0
rolling_vol_60d     0
rolling_vol_252d    0
cum_max             0
drawdown            0
running_peak        0
tomorrow_close      0
target              0
dtype: int64


# Final Output

The processed dataset includes:

- Clean OHLCV data  
- Engineered financial features  
- Target classification label  

This dataset serves as the **single source of truth** for:

- Machine learning models  
- R-based statistical analysis  
- Power BI dashboard visualization  


In [32]:
# Export the fully processed, ML-ready dataset.
# This file is the single source of truth for the R analysis and Power BI dashboard.
output_file = "processed_stock_data.csv"
df.to_csv(output_file, index=False)
print(f"✓ Processed dataset saved → {output_file}")
print(f"  Rows: {len(df):,}  |  Columns: {len(df.columns)}")

✓ Processed dataset saved → processed_stock_data.csv
  Rows: 11,310  |  Columns: 26
